In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Stock Price Predictor Pro
## Advanced ML Powered Stock Analysis & Prediction

**Features in this notebook:**
-  Live stock data via `yfinance`
-  Rich technical indicators (RSI, MACD, Bollinger Bands, Moving Averages)
-  LSTM deep learning model for 7 day price forecasting
-  Random Forest + Linear Regression ensemble comparison
-  Interactive Plotly candlestick & signal charts
- 🟢🔴 Buy / Sell signal generation
-  Model performance metrics dashboard


In [2]:
!pip install yfinance plotly scikit-learn pandas numpy matplotlib -q

In [3]:
#Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('All packages loaded successfully!')

All packages loaded successfully!
TensorFlow version: 2.19.0


## 1.
 Fetch Live Stock Data
We download 3 years of daily OHLCV data from Yahoo Finance.

In [4]:
#Configuration
TICKER   = 'AAPL'    # Change to any ticker: 'TSLA', 'GOOGL', 'MSFT', etc.
PERIOD   = '3y'      # Data period
INTERVAL = '1d'      # Daily data (1 day)
FORECAST_DAYS = 7    # Number of future days to predict

#Download
print(f' Downloading {TICKER} data...')
raw = yf.download(TICKER, period=PERIOD, interval=INTERVAL, auto_adjust=True)

# Flatten MultiIndex columns if present
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
df.dropna(inplace=True)

print(f'{len(df)} trading days loaded ({df.index[0].date()} → {df.index[-1].date()})')
df.tail()

[*********************100%***********************]  1 of 1 completed

753 trading days loaded (2023-04-10 → 2026-04-09)


Price,Open,High,Low,Close,Volume
Date,,,,,
2026-04-02,254.199997,256.130005,250.649994,255.919998,31289400
2026-04-06,256.510010,262.160004,256.459991,258.859985,29329900
2026-04-07,256.160004,256.200012,245.699997,253.500000,62148000
2026-04-08,258.450012,259.750000,256.529999,258.899994,41008200
2026-04-09,259.369995,260.029999,256.070007,259.480011,11986012


## 2. Feature Engineering Technical Indicators
We compute a rich set of indicators used by professional traders.

In [5]:
def add_technical_indicators(df):
    """Add a comprehensive set of technical indicators to the dataframe."""

    #  Moving Averages
    df['SMA_20']  = df['Close'].rolling(20).mean()
    df['SMA_50']  = df['Close'].rolling(50).mean()
    df['SMA_200'] = df['Close'].rolling(200).mean()
    df['EMA_12']  = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26']  = df['Close'].ewm(span=26, adjust=False).mean()

    # Bollinger Bands
    rolling_std = df['Close'].rolling(20).std()
    df['BB_Upper'] = df['SMA_20'] + 2 * rolling_std
    df['BB_Lower'] = df['SMA_20'] - 2 * rolling_std
    df['BB_Width'] = df['BB_Upper'] - df['BB_Lower']

    # MACD
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist']   = df['MACD'] - df['MACD_Signal']

    # RSI(14 period)
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    rs    = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # Average True Range(ATR)
    high_low  = df['High'] - df['Low']
    high_prev = (df['High'] - df['Close'].shift()).abs()
    low_prev  = (df['Low']  - df['Close'].shift()).abs()
    df['ATR'] = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1).rolling(14).mean()

    # Price Momentum & Volatility
    df['Return_1d']  = df['Close'].pct_change(1)
    df['Return_5d']  = df['Close'].pct_change(5)
    df['Volatility'] = df['Return_1d'].rolling(20).std()

    # Volume Indicators
    df['Volume_SMA'] = df['Volume'].rolling(20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']

    # Target: Next Day Close
    df['Target'] = df['Close'].shift(-1)

    return df


df = add_technical_indicators(df)
df.dropna(inplace=True)

print(f'Technical indicators computed. Dataset shape: {df.shape}')
print(f'Features available: {list(df.columns)}')

Technical indicators computed. Dataset shape: (553, 24)
Features available: ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'SMA_200', 'EMA_12', 'EMA_26', 'BB_Upper', 'BB_Lower', 'BB_Width', 'MACD', 'MACD_Signal', 'MACD_Hist', 'RSI', 'ATR', 'Return_1d', 'Return_5d', 'Volatility', 'Volume_SMA', 'Volume_Ratio', 'Target']


## 3. 🟢🔴 Buy / Sell Signal Generation
Signals are based on RSI + MACD crossover + Bollinger Band rules.

In [6]:
def generate_signals(df):
    """Generate trading signals using multiple technical indicator rules."""
    df = df.copy()

    # RSI signals
    rsi_buy  = df['RSI'] < 35
    rsi_sell = df['RSI'] > 65

    # MACD crossover signals
    macd_cross_up   = (df['MACD'] > df['MACD_Signal']) & (df['MACD'].shift(1) <= df['MACD_Signal'].shift(1))
    macd_cross_down = (df['MACD'] < df['MACD_Signal']) & (df['MACD'].shift(1) >= df['MACD_Signal'].shift(1))

    # Bollinger Band breakout
    bb_buy  = df['Close'] < df['BB_Lower']
    bb_sell = df['Close'] > df['BB_Upper']

    # Combined signal (at least 2 of 3 conditions agree)
    buy_score  = rsi_buy.astype(int)  + macd_cross_up.astype(int)  + bb_buy.astype(int)
    sell_score = rsi_sell.astype(int) + macd_cross_down.astype(int) + bb_sell.astype(int)

    df['Signal'] = 0  # Hold
    df.loc[buy_score  >= 2, 'Signal'] =  1   # Buy
    df.loc[sell_score >= 2, 'Signal'] = -1   # Sell

    return df


df = generate_signals(df)

buy_count  = (df['Signal'] ==  1).sum()
sell_count = (df['Signal'] == -1).sum()
hold_count = (df['Signal'] ==  0).sum()

print(f'🟢 Buy signals:  {buy_count}')
print(f'🔴 Sell signals: {sell_count}')
print(f'⚪ Hold signals: {hold_count}')

🟢 Buy signals:  26
🔴 Sell signals: 30
⚪ Hold signals: 497


## 4. Interactive Plotly Charts

In [7]:
def plot_price_with_signals(df, ticker):
    """Candlestick chart with moving averages,Bollinger Bands,and buy/sell signals."""

    # Use last 180 trading days for clarity
    plot_df = df.tail(180).copy()

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.6, 0.2, 0.2],
        vertical_spacing=0.04,
        subplot_titles=(f'{ticker} Price + Signals', 'MACD', 'RSI')
    )

    # Row 1: Candlestick + Indicators
    fig.add_trace(go.Candlestick(
        x=plot_df.index,
        open=plot_df['Open'], high=plot_df['High'],
        low=plot_df['Low'],   close=plot_df['Close'],
        name='OHLC', increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ), row=1, col=1)

    # Bollinger Bands
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['BB_Upper'], name='BB Upper',
                             line=dict(color='rgba(100,100,255,0.4)', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['BB_Lower'], name='BB Lower',
                             line=dict(color='rgba(100,100,255,0.4)', dash='dot'),
                             fill='tonexty', fillcolor='rgba(100,100,255,0.05)'), row=1, col=1)

    # Moving Averages
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_20'],  name='SMA 20',
                             line=dict(color='orange', width=1.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['SMA_50'],  name='SMA 50',
                             line=dict(color='purple', width=1.5)), row=1, col=1)

    # Buy / Sell signals
    buy_signals  = plot_df[plot_df['Signal'] ==  1]
    sell_signals = plot_df[plot_df['Signal'] == -1]

    fig.add_trace(go.Scatter(x=buy_signals.index,  y=buy_signals['Low']  * 0.98,
                             mode='markers', name='Buy Signal',
                             marker=dict(symbol='triangle-up', size=12, color='#00e676')), row=1, col=1)
    fig.add_trace(go.Scatter(x=sell_signals.index, y=sell_signals['High'] * 1.02,
                             mode='markers', name='Sell Signal',
                             marker=dict(symbol='triangle-down', size=12, color='#ff1744')), row=1, col=1)

    # Row 2: MACD
    colors = ['#26a69a' if v >= 0 else '#ef5350' for v in plot_df['MACD_Hist']]
    fig.add_trace(go.Bar(x=plot_df.index, y=plot_df['MACD_Hist'],
                         name='MACD Hist', marker_color=colors), row=2, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['MACD'],
                             name='MACD', line=dict(color='#2196F3', width=1.5)), row=2, col=1)
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['MACD_Signal'],
                             name='Signal Line', line=dict(color='#FF9800', width=1.5)), row=2, col=1)

    # Row 3: RSI
    fig.add_trace(go.Scatter(x=plot_df.index, y=plot_df['RSI'],
                             name='RSI', line=dict(color='#9c27b0', width=2)), row=3, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='red',   row=3, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='green', row=3, col=1)

    fig.update_layout(
        title=f'{ticker} — Technical Analysis Dashboard',
        template='plotly_dark',
        height=900,
        showlegend=True,
        xaxis_rangeslider_visible=False
    )
    fig.show()


plot_price_with_signals(df, TICKER)

## 5. ML Models — Feature Preparation

In [8]:
# Feature columns for ML
FEATURE_COLS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
    'BB_Upper', 'BB_Lower', 'BB_Width',
    'MACD', 'MACD_Signal', 'MACD_Hist',
    'RSI', 'ATR',
    'Return_1d', 'Return_5d', 'Volatility',
    'Volume_Ratio'
]

X = df[FEATURE_COLS].values
y = df['Target'].values

# 80/20 chronological split (NO shuffle — time-series)
split_idx = int(len(X) * 0.80)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f'Train samples: {len(X_train)}   Test samples: {len(X_test)}')
print(f'Feature count: {X.shape[1]}')

Train samples: 442   Test samples: 111
Feature count: 21


## 6. 📉 Baseline Models — Linear Regression & Random Forest

In [9]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f'{name:<25}  MAE: ${mae:.2f}   RMSE: ${rmse:.2f}   R²: {r2:.4f}')
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2}


# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

print('' * 60)
print('MODEL COMPARISON')
print('' * 60)
lr_metrics = evaluate('Linear Regression',   y_test, lr_preds)
rf_metrics = evaluate('Random Forest (200)', y_test, rf_preds)
print('' * 60)


MODEL COMPARISON

Linear Regression          MAE: $2.51   RMSE: $3.55   R²: 0.8654
Random Forest (200)        MAE: $6.95   RMSE: $8.26   R²: 0.2706



## 7. Gradient Boosting Model
Gradient Boosting builds an ensemble of decision trees sequentially, each correcting the errors of the previous one. It is highly accurate for tabular/time-series data and requires no special Python version.

In [10]:
# Scale features for Gradient Boosting
scaler_X = MinMaxScaler()
X_all_scaled = scaler_X.fit_transform(X)

X_train_sc = X_all_scaled[:split_idx]
X_test_sc  = X_all_scaled[split_idx:]

print(f'GB train shape: {X_train_sc.shape}   GB test shape: {X_test_sc.shape}')

LSTM train shape: (394, 60, 21)   LSTM test shape: (99, 60, 21)


In [11]:
# Build & train Gradient Boosting model
gb_model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_train_sc, y_train)
print('Gradient Boosting model trained!')
print(f'Number of estimators: {gb_model.n_estimators_}')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        76,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 139,169 (543.63 KB)

 Trainable params: 139,169 (543.63 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Gradient Boosting Predictions
gb_preds  = gb_model.predict(X_test_sc)
gb_actual = y_test

gb_metrics = evaluate('Gradient Boosting', gb_actual, gb_preds)
print('-' * 60)

Epoch 1/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 13s 169ms/step - loss: 0.0261 - val_loss: 0.0234
Epoch 2/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - loss: 0.0090 - val_loss: 0.0247
Epoch 3/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - loss: 0.0051 - val_loss: 0.0208
Epoch 4/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - loss: 0.0052 - val_loss: 0.0145
Epoch 5/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - loss: 0.0050 - val_loss: 0.0175
Epoch 6/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 117ms/step - loss: 0.0047 - val_loss: 0.0095
Epoch 7/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 200ms/step - loss: 0.0047 - val_loss: 0.0057
Epoch 8/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 126ms/step - loss: 0.0061 - val_loss: 0.0058
Epoch 9/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 115ms/step - loss: 0.0058 - val_loss: 0.0181
Epoch 10/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 118ms/step - loss: 0.0045 - val_loss: 0.0199
Epoch 11/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - loss: 0.0035 - val_loss: 0.0141
Epoch 12/60
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 114ms/ste

In [13]:
# Gradient Boosting — Feature Importance Plot
feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': gb_model.feature_importances_})
feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)

fig_imp = go.Figure(go.Bar(
    x=feat_df['Importance'], y=feat_df['Feature'],
    orientation='h',
    marker=dict(color=feat_df['Importance'], colorscale='Viridis')
))
fig_imp.update_layout(
    title='Gradient Boosting — Top 15 Feature Importances',
    template='plotly_dark', height=450
)
fig_imp.show()

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 161ms/step
LSTM (3-layer)             MAE: $12.60   RMSE: $14.79   R²: -1.1687



## 8. Model Comparison — Actual vs Predicted

In [14]:
# Model Comparison Chart
test_dates = df.index[split_idx:].tolist()

fig_cmp = go.Figure()
fig_cmp.add_trace(go.Scatter(x=test_dates, y=y_test,    mode='lines',
                             name='Actual Close',       line=dict(color='white',   width=2)))
fig_cmp.add_trace(go.Scatter(x=test_dates, y=rf_preds,  mode='lines',
                             name='Random Forest',      line=dict(color='#FF9800', width=1.5)))
fig_cmp.add_trace(go.Scatter(x=test_dates, y=lr_preds,  mode='lines',
                             name='Linear Reg',         line=dict(color='#2196F3', width=1.5)))
fig_cmp.add_trace(go.Scatter(x=test_dates, y=gb_preds,  mode='lines',
                             name='Gradient Boosting',  line=dict(color='#00e676', width=2)))

fig_cmp.update_layout(
    title=f'{TICKER}  Model Predictions vs Actual Prices',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_dark', height=500
)
fig_cmp.show()

## 9. 7-Day Future Price Forecast

In [15]:
def forecast_next_n_days(model, last_row, scaler_X, n_days=7):
    """
    Iterative multi-step forecast using Gradient Boosting.
    Each day we predict the next close, then feed it back as the next input.
    """
    preds     = []
    row       = last_row.copy()
    close_idx = FEATURE_COLS.index('Close')

    for _ in range(n_days):
        row_scaled  = scaler_X.transform(row.reshape(1, -1))
        pred_price  = model.predict(row_scaled)[0]
        preds.append(pred_price)
        row[close_idx] = pred_price  # feed prediction back

    return preds


# Use last raw (unscaled) row as starting point
last_row     = X[-1].copy()
future_prices = forecast_next_n_days(gb_model, last_row, scaler_X, FORECAST_DAYS)

# Date range for future predictions (skip weekends)
last_date    = df.index[-1]
future_dates = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS)

# Plot Forecast
hist_tail = df['Close'].tail(60)

fig_fc = go.Figure()
fig_fc.add_trace(go.Scatter(x=hist_tail.index, y=hist_tail.values,
                            mode='lines', name='Historical Close',
                            line=dict(color='#90caf9', width=2)))
fig_fc.add_trace(go.Scatter(x=future_dates, y=future_prices,
                            mode='lines+markers', name='GB Forecast',
                            line=dict(color='#ffd54f', width=2, dash='dash'),
                            marker=dict(size=8, color='#ffd54f')))
upper = [p * 1.02 for p in future_prices]
lower = [p * 0.98 for p in future_prices]
fig_fc.add_trace(go.Scatter(
    x=list(future_dates) + list(future_dates[::-1]),
    y=upper + lower[::-1],
    fill='toself', fillcolor='rgba(255,213,79,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='±2% Confidence'
))

fig_fc.update_layout(
    title=f'{TICKER} — {FORECAST_DAYS}-Day Gradient Boosting Price Forecast',
    xaxis_title='Date', yaxis_title='Price (USD)',
    template='plotly_dark', height=450
)
fig_fc.show()

print(' Forecasted Prices:')
for d, p in zip(future_dates, future_prices):
    print(f'  {d.date()}  →  ${p:.2f}')

 Forecasted Prices:
  2026-04-09  →  $246.50
  2026-04-10  →  $246.80
  2026-04-13  →  $247.10
  2026-04-14  →  $247.40
  2026-04-15  →  $247.66
  2026-04-16  →  $247.89
  2026-04-17  →  $248.10


## 10.  Final Performance Dashboard

In [16]:
# Summary Bar Chart
models      = ['Linear Regression', 'Random Forest', 'Gradient Boosting']
all_preds   = [lr_preds, rf_preds, gb_preds]
all_actuals = [y_test,   y_test,   gb_actual]

metrics_df = pd.DataFrame([
    {
        'Model': m,
        'MAE':  mean_absolute_error(a, p),
        'RMSE': np.sqrt(mean_squared_error(a, p)),
        'R²':   r2_score(a, p)
    }
    for m, a, p in zip(models, all_actuals, all_preds)
])

print(' MODEL PERFORMANCE SUMMARY ')
print(metrics_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

fig_bar = make_subplots(rows=1, cols=3,
                        subplot_titles=('Mean Absolute Error (lower=better)',
                                        'RMSE (lower=better)',
                                        'R² Score (higher=better)'))

colors_bar = ['#ef5350', '#FF9800', '#00e676']
for col, metric in enumerate(['MAE', 'RMSE', 'R²'], start=1):
    fig_bar.add_trace(
        go.Bar(x=metrics_df['Model'], y=metrics_df[metric],
               marker_color=colors_bar, name=metric,
               text=[f'{v:.2f}' for v in metrics_df[metric]],
               textposition='outside'),
        row=1, col=col
    )

fig_bar.update_layout(
    title='Model Performance Comparison',
    template='plotly_dark', height=400,
    showlegend=False
)
fig_bar.show()

 MODEL PERFORMANCE SUMMARY 
            Model     MAE    RMSE      R²
Linear Regression  2.5129  3.5501  0.8654
    Random Forest  6.9543  8.2630  0.2706
             LSTM 12.6012 14.7938 -1.1687


In [17]:
# Feature Importance (Random Forest)
importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=True).tail(15)

fig_imp = go.Figure(go.Bar(
    x=feat_df['Importance'], y=feat_df['Feature'],
    orientation='h',
    marker=dict(color=feat_df['Importance'], colorscale='Viridis')
))
fig_imp.update_layout(
    title='Random Forest  Top 15 Feature Importances',
    template='plotly_dark', height=500
)
fig_imp.show()

## Key Insights

| Aspect | Details |
|--------|---------|
| **Data** | 3 years daily OHLCV from Yahoo Finance |
| **Features** | 21 technical indicators (SMA, EMA, BB, MACD, RSI, ATR, momentum, volatility) |
| **Models** | Linear Regression, Random Forest, Gradient Boosting |
| **GB Settings** | 300 estimators, depth 5, learning rate 0.05, subsample 0.8 |
| **Forecast Horizon** | 7 trading days ahead |
| **Buy/Sell Signals** | Multi-indicator consensus (RSI + MACD crossover + Bollinger Band breakout) |
| **No TensorFlow** | Works on Python 3.14+ and all Streamlit/Colab environments |